In [69]:
# create KG from yt data


In [70]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, XSD, Literal
from rdflib.namespace import DCTERMS, SKOS

g = Graph()
# Namespaces
SCHEMA = Namespace("http://schema.org/")
WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")
DANCE = Namespace("http://example.org/dance/")

# Graph setup
g = Graph()
g.bind("schema", SCHEMA)
g.bind("wd", WD)
g.bind("wdt", WDT)
g.bind("dance", DANCE)
g.bind("dcterms", DCTERMS)
g.bind("skos", SKOS)
g.bind("owl", OWL)



In [71]:
# Helpers

def add_class(uri, label, parent=None, comment=None):
    g.add((uri, RDF.type, OWL.Class))
    g.add((uri, RDFS.label, Literal(label, lang="en")))
    if parent is not None:
        g.add((uri, RDFS.subClassOf, parent))
    if comment is not None:
        g.add((uri, RDFS.comment, Literal(comment, lang="en")))


def add_obj_prop(uri, label, domain, range_):
    g.add((uri, RDF.type, OWL.ObjectProperty))
    g.add((uri, RDFS.label, Literal(label, lang="en")))
    g.add((uri, RDFS.domain, domain))
    g.add((uri, RDFS.range, range_))
    #if superprop is not None:
    #    g.add((uri, RDFS.subPropertyOf, superprop))
    #if equiv is not None:
     #   g.add((uri, OWL.equivalentProperty, equiv))


def add_data_prop(uri, label, domain, range_=XSD.string):
    g.add((uri, RDF.type, OWL.DatatypeProperty))
    g.add((uri, RDFS.label, Literal(label, lang="en")))
    g.add((uri, RDFS.domain, domain))
    g.add((uri, RDFS.range, range_))
    #if superprop is not None:
    #    g.add((uri, RDFS.subPropertyOf, superprop))


def add_named_instance(instance_uri, class_uri, name):
    g.add((instance_uri, RDF.type, class_uri))
    g.add((instance_uri, SCHEMA.name, Literal(name, lang="en")))

def ensure_named_instance(class_uri, value, prefix=None):

    key = str(value).strip().replace(" ", "_")
    node = DANCE[key]
    g.add((node, RDF.type, class_uri))
    g.add((node, SCHEMA.name, Literal(str(value), lang="en")))
    return node


In [72]:
# add yt ontology
add_class(DANCE.YTLINK,"has corresponding yt video", parent=SCHEMA.VideoObject)
#add_obj_prop(DANCE.hasYTLink, "has yt link", DANCE.DanceRecord, DANCE.hasYTLink)

add_data_prop(SCHEMA.title, "video title", DANCE.YTLINK, XSD.string)
add_data_prop(WD.Q28659447, "comment Sentiment", DANCE.YTLINK, XSD.string)
add_data_prop(SCHEMA.uploadDate,"upload date", DANCE.YTLINK, XSD.date )
add_data_prop(SCHEMA.desription,"description", DANCE.YTLINK, XSD.string)
add_data_prop(SCHEMA.UserLikes, "likes on video", DANCE.YTLINK, XSD.integer)
add_data_prop(SCHEMA.interactionCount,"view_count", DANCE.YTLINK, XSD.integer)
add_data_prop(SCHEMA.duration, "duration", DANCE.YTLINK, XSD.duration)



In [73]:
# ── Load filtered video CSV ────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

CSV_PATH = Path("../data/yt_data/dance_videos_filtered.csv")
df = pd.read_csv(CSV_PATH)

# Coerce numeric columns
df["view_count"]                     = pd.to_numeric(df["view_count"],                     errors="coerce")
df["like_count"]                     = pd.to_numeric(df["like_count"],                     errors="coerce")
df["comment_count"]                  = pd.to_numeric(df["comment_count"],                  errors="coerce")
df["num_comments_retrieved"]         = pd.to_numeric(df["num_comments_retrieved"],         errors="coerce")
df["comment_sentiment_avg_compound"] = pd.to_numeric(df["comment_sentiment_avg_compound"], errors="coerce")

print(f"Loaded {len(df)} rows from {CSV_PATH}")
df.head(3)


Loaded 91 rows from ../data/yt_data/dance_videos_filtered.csv


,dance_style,dance_type,search_query,video_id,video_url,title,description,channel,published_at,view_count,like_count,comment_count,duration,num_comments_retrieved,comment_sentiment,comment_sentiment_avg_compound,comment_sentiment_comment_count
0,American Rhythm,Latin dance / Rhythm,Latin dance / Rhythm American Rhythm tutorial,LZjBD8ZZAOo,https://www.youtube.com/watch?v=LZjBD8ZZAOo,How To Count Samba (For Beginners) - Samba Rhy...,Join as a member (Free Trial): https://www.pas...,Passion4dancing,2018-03-27T21:55:24Z,157764,2018,46,PT5M22S,10,positive,0.7,10.0
1,American Tribal Style,Belly dance,Belly dance American Tribal Style tutorial,MFZG49yVtco,https://www.youtube.com/watch?v=MFZG49yVtco,"American Tribal Style (ATS) by Sirin Tribe, De...","Центр трайбл-культуры в Санкт-Петербурге, колл...",Sirin Tribe,2017-01-15T18:53:33Z,267698,3242,71,PT4M59S,10,positive,1.0,10.0
2,Argentine tango,Latin dance,Latin dance Argentine tango tutorial,qrdE4jqXq_k,https://www.youtube.com/watch?v=qrdE4jqXq_k,Beginner Argentine Tango Basics,"In this video, Helen Wang is going to teach Ar...",Helen Wang Tango,2021-01-31T16:30:04Z,675811,11773,135,PT6M20S,10,positive,1.0,10.0


In [74]:
# ── Load existing dance KG so we can link DanceRecords ────────────────────────
from rdflib import Graph as RDFGraph, URIRef

MAIN_KG_PATH = Path("dance_kg_with_data.ttl")

main_g = RDFGraph()
main_g.parse(MAIN_KG_PATH, format="turtle")
print(f"Main KG loaded — {len(main_g)} triples")

# Build a lookup: (dance_style_name, dance_type_name) → DanceRecord URI
# Records carry schema1:name like "American - Jazz dance"
DANCE_NS  = "http://example.org/dance/"
SCHEMA_NS = "http://schema.org/"

record_lookup: dict[tuple[str, str], URIRef] = {}

sparql = """
PREFIX dance:   <http://example.org/dance/>
PREFIX schema1: <http://schema.org/>

SELECT DISTINCT ?style ?styleName WHERE {
  ?style a dance:DanceStyle;
         schema1:name ?styleName .
}
"""
DANCE_NS = "http://example.org/dance/"
for row in main_g.query(sparql):
    key = str(row.styleName).strip()
    record_lookup[key] = row.style

print(f"DanceStyle entries indexed: {len(record_lookup)}")
print("Keys:", sorted(record_lookup.keys()))


Main KG loaded — 5888 triples
DanceStyle entries indexed: 127
Keys: ['American Rhythm', 'American Tribal Style', 'Argentine tango', 'Bachata', 'Baladi', 'Balboa', 'Ballet', 'Baroque dance', 'Bhangra', 'Bharatanatyam', 'Big Apple', 'Black Bottom', 'Blues dance', 'Bolero', 'Bollywood dance', 'Boogie-woogie', 'Breaking', 'Bugg', 'Bunny hop', 'Carolina Shag', 'Cha Cha', 'Cham dance', 'Charleston', 'Circle dance', 'Collegiate Shag', 'Conga line', 'Contact improvisation', 'Contemporary dance', 'Country dance', 'Cumbia', 'Dabkeh', 'Danza', 'Drametse Ngacham', 'Duranguense', 'East Coast Swing', 'Ecstatic dance', 'Electric Boogaloo', 'Electro dance', 'Fire dance', 'Flamenco', 'Footwork', 'Forró', 'Frug', 'Gangsta Walking', 'Go-go dance', 'Haka', 'Hand Jive', 'Hand dancing', 'Harlem shake', 'Hip-hop dance', 'House dance', 'Hustle', 'Ice Dance', 'International Latin', 'Interpretive dance', 'Jazz dance', 'Jitterbug', 'Jive', 'Jumpstyle', 'Kagura', 'Kawleeya', 'Khaleeji', 'Krumping', 'Lambada', 'Li

In [75]:
# ── Add object property declaration to this graph ─────────────────────────────
add_obj_prop(DANCE.hasYTLink, "has yt link", DANCE.DanceStyle, DANCE.YTLINK)


In [76]:
# ── Populate one YTLINK instance per video row ────────────────────────────────
linked   = 0
unlinked = 0

for _, row in df.iterrows():
    vid_id = str(row["video_id"]).strip()

    # URI for this video node
    video_uri = DANCE[f"video_{vid_id}"]

    # Type
    g.add((video_uri, RDF.type, DANCE.YTLINK))

    # Data properties — only those defined in the ontology
    g.add((video_uri, SCHEMA.title,Literal(str(row["title"]),       datatype=XSD.string)))
    g.add((video_uri, SCHEMA.description,Literal(str(row["description"]), datatype=XSD.string)))

    if pd.notna(row["published_at"]):
        g.add((video_uri, SCHEMA.uploadDate,Literal(str(row["published_at"])[:10], datatype=XSD.date)))

    if pd.notna(row["like_count"]):
        g.add((video_uri, SCHEMA.UserLikes,Literal(int(row["like_count"]),        datatype=XSD.integer)))

    if pd.notna(row["view_count"]):
        g.add((video_uri, SCHEMA.interactionCount, Literal(int(row["view_count"]),datatype=XSD.integer)))

    if pd.notna(row["comment_sentiment"]):
        g.add((video_uri, WD.Q28659447,Literal(str(row["comment_sentiment"]), datatype=XSD.string)))

    if pd.notna(row["duration"]):
        g.add((video_uri, SCHEMA.duration,Literal(str(row["duration"]),datatype=XSD.string)))


    # ── Link DanceStyle → video ────────────────────────────────────────────────
    dstyle = str(row["dance_style"]).strip()
    style_uri = record_lookup.get(dstyle)

    if style_uri:
        g.add((style_uri, DANCE.hasYTLink, video_uri))
        linked += 1
    else:
        print(f"  No DanceStyle found for: '{dstyle}'")
        unlinked += 1

print(f"\n✓ Videos added : {linked + unlinked}")
print(f"  Linked to DanceRecord : {linked}")
print(f"  Unlinked (no match)   : {unlinked}")
print(f"  Triples in YT graph   : {len(g)}")



✓ Videos added : 91
  Linked to DanceRecord : 91
  Unlinked (no match)   : 0
  Triples in YT graph   : 826


In [77]:
# ── Serialise ─────────────────────────────────────────────────────────────────
OUT_PATH = Path("dance_yt_kg.ttl")
g.serialize(destination=str(OUT_PATH), format="turtle")
print(f"Saved → {OUT_PATH}  ({len(g)} triples)")


Saved → dance_yt_kg.ttl  (826 triples)


In [78]:
# ── Merge main KG + YT KG into one graph ──────────────────────────────────────
merged_g = RDFGraph()

# Copy namespace bindings
for prefix, ns in g.namespaces():
    merged_g.bind(prefix, ns)

merged_g.parse(MAIN_KG_PATH, format="turtle")
print(f"Main KG triples : {len(merged_g)}")

merged_g.parse(str(OUT_PATH), format="turtle")
print(f"After merge     : {len(merged_g)} triples")

MERGED_PATH = Path("dance_kg_merged_with_yt.ttl")
merged_g.serialize(destination=str(MERGED_PATH), format="turtle")
print(f"Saved → {MERGED_PATH}")



Main KG triples : 5888
After merge     : 6706 triples
Saved → dance_kg_merged_with_yt.ttl
